In [29]:
import csv
import io
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Optional, Union

import requests
from Bio import SeqIO

UNIPROT_BASE_URL = "https://rest.uniprot.org/uniprotkb"
ENSEMBL_BASE_URL = "https://rest.ensembl.org"

UNIPROT_RE_TEXT = re.compile(
    r"(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]{5})"
)
ENSEMBL_RE_TEXT = re.compile(r"ENS[A-Z0-9]+[0-9]+")


class FastaDBParser:
   
    @staticmethod
    def _find_seqkit_executable() -> Optional[str]:
        
        conda_prefix = os.environ.get("CONDA_PREFIX", "")
        candidates = [
            shutil.which("seqkit"),
            str(Path(sys.prefix) / "bin" / "seqkit"),
            str(Path(conda_prefix) / "bin" / "seqkit") if conda_prefix else None,
            "/opt/homebrew/bin/seqkit",
            "/usr/local/bin/seqkit",
            str(Path.home() / "bin" / "seqkit"),
        ]
        for p in candidates:
            if p and Path(p).is_file():
                return p
        return None

    def __init__(self, seqkit_bin: Optional[str] = None):
        self.seqkit_bin = seqkit_bin or self._find_seqkit_executable() or "seqkit"

    @staticmethod
    def __strip_ansi(s: str) -> str:
        return re.sub(r"\x1b\[[0-9;]*m", "", s or "")

    def __http_function(self, endpoint, method="GET", headers=None, params=None):
        if method == "GET":
            return requests.get(endpoint, headers=headers, params=params, timeout=60)
        raise ValueError("unsupported HTTP method")

    def __get_uniprot(self, accession):
        endpoint = f"{UNIPROT_BASE_URL}/{accession}.json"
        http_args = {"headers": {"Accept": "application/json"}}
        return self.__http_function(endpoint, **http_args)

    def __uniprot_parse_response(self, resp):
        status = resp.status_code
        if status != 200:
            try:
                data = resp.json()
                msg_list = data.get("messages")
                if isinstance(msg_list, list) and msg_list:
                    message = msg_list[0]
                else:
                    message = data.get("error") or json.dumps(data)
            except Exception:
                message = resp.text
            return {"error": message}

        data = resp.json()
        accession = data.get("primaryAccession") or data.get("uniProtkbId")
        if not accession:
            return {"error": "missing accession"}

        organism = None
        org_block = data.get("organism") or {}
        if isinstance(org_block, dict):
            organism = org_block.get("scientificName")

        genes = data.get("genes") or []

        seq_block = data.get("sequence") or {}
        if not isinstance(seq_block, dict):
            seq_block = {}
        sequence_info = {
            "value": seq_block.get("value"),
            "length": seq_block.get("length"),
            "molWeight": seq_block.get("molWeight"),
            "crc64": seq_block.get("crc64"),
            "md5": seq_block.get("md5"),
        }

        entry_type = data.get("entryType") or "protein"

        return {
            accession: {
                "organism": organism,
                "geneInfo": genes,
                "sequenceInfo": sequence_info,
                "type": entry_type,
            }
        }

    def __get_ensembl(self, id_):
        endpoint = f"{ENSEMBL_BASE_URL}/lookup/id/{id_}"
        http_args = {"headers": {"Accept": "application/json"}, "params": {"expand": 0}}
        return self.__http_function(endpoint, **http_args)

    def __ensembl_parse_response(self, resp):
        status = resp.status_code
        if status != 200:
            try:
                data = resp.json()
                message = data.get("error") or json.dumps(data)
            except Exception:
                message = resp.text
            return {"error": message}

        data = resp.json()
        ensembl_id = data.get("id")
        if not ensembl_id:
            return {"error": "missing id"}

        fields = [
            "object_type",
            "species",
            "assembly_name",
            "biotype",
            "display_name",
            "id",
            "db_type",
            "description",
            "canonical_transcript",
            "source",
        ]

        parsed = {key: data.get(key) for key in fields}
        return {ensembl_id: parsed}

    @staticmethod
    def __parse_seqkit_tsv(stdout: str) -> dict:
        reader = csv.DictReader(io.StringIO(stdout), delimiter="\t")
        rows = list(reader)
        if not rows:
            raise ValueError("seqkit stats: нет строк данных после заголовка")
        return rows[0]

    def run_seqkit_stats(self, fasta_path: Union[str, Path]) -> dict:
        """Возвращает {'ok': True, 'molecule_type': ..., 'stats': {...}} либо {'ok': False, 'error': ...}."""
        path = str(fasta_path)
        try:
            proc = subprocess.run(
                [self.seqkit_bin, "stats", "-T", path],
                capture_output=True,
                text=True,
            )
        except FileNotFoundError:
            hint_conda = str(Path(sys.prefix) / "bin" / "seqkit")
            return {
                "ok": False,
                "error": (
                    f"Не найден seqkit ({self.seqkit_bin!r}). Установите: "
                    "`conda install -y -c bioconda seqkit` в том же env, что и ядро Jupyter, "
                    "или скачайте бинарник с GitHub (релизы seqkit). "
                    f"Часто помогает явный путь: FastaDBParser(seqkit_bin={hint_conda!r}) "
                    "(или вывод `which seqkit` в терминале после conda activate)."
                ),
            }

        err = self.__strip_ansi(proc.stderr or "").strip()
        if err:
            return {"ok": False, "error": err, "returncode": proc.returncode}

        try:
            stats_row = self.__parse_seqkit_tsv(proc.stdout)
        except ValueError as e:
            return {"ok": False, "error": str(e), "raw_stdout": proc.stdout}

        mol = (stats_row.get("type") or "").strip()
        if not mol:
            return {"ok": False, "error": "Не удалось прочитать колонку type из seqkit stats"}

        return {"ok": True, "molecule_type": mol, "stats": stats_row}

    @staticmethod
    def __first_uniprot(description: str) -> Optional[str]:
        m = UNIPROT_RE_TEXT.search(description or "")
        return m.group(0) if m else None

    @staticmethod
    def __first_ensembl(description: str) -> Optional[str]:
        m = ENSEMBL_RE_TEXT.search(description or "")
        return m.group(0) if m else None

    def process_fasta(self, fasta_path: Union[str, Path]) -> dict:
       
        sk = self.run_seqkit_stats(fasta_path)
        if not sk.get("ok"):
            return {"error": sk.get("error", "unknown"), "seqkit": sk}

        mol = sk["molecule_type"]
        use_uniprot = mol == "Protein"
        use_ensembl = mol in ("DNA", "RNA")

        if mol == "Unlimit":
            first = next(SeqIO.parse(str(fasta_path), "fasta"), None)
            if first is None:
                return {"error": "Пустой FASTA", "seqkit": sk}
            letters = set(str(first.seq).upper()) - set("-. ")
            dna_like = letters <= set("ACGTUNRYSWKMBDHV")
            use_uniprot = not dna_like
            use_ensembl = dna_like

        sequences_out = []
        for record in SeqIO.parse(str(fasta_path), "fasta"):
            desc = record.description
            seq = str(record.seq)
            db_name = None
            db_info = None

            if use_uniprot:
                uid = self.__first_uniprot(desc)
                db_name = "UniProt"
                if uid:
                    db_info = self.__uniprot_parse_response(self.__get_uniprot(uid))
                else:
                    db_info = {"error": "В описании не найден accession UniProt"}
            elif use_ensembl:
                eid = self.__first_ensembl(desc)
                db_name = "Ensembl"
                if eid:
                    db_info = self.__ensembl_parse_response(self.__get_ensembl(eid))
                else:
                    db_info = {"error": "В описании не найден ID Ensembl"}
            else:
                db_name = None
                db_info = {
                    "error": f"Тип '{mol}' не поддерживается для автоматического выбора БД"
                }

            sequences_out.append(
                {
                    "description": desc,
                    "sequence": seq,
                    "db_name": db_name,
                    "db_info": db_info,
                }
            )

        return {"seqkit": sk, "sequences": sequences_out}


def pprint_result(data: dict) -> None:
    print(json.dumps(data, indent=2, ensure_ascii=False))

In [30]:

BASE = Path(".")
parser = FastaDBParser()

test_files = [
    BASE / "uniprot_download.fasta",
    BASE / "ensembl_download_1.fasta",
    BASE / "ensembl_download_2.fasta",
]

results = {}
for fp in test_files:
    print("\n", "=" * 72)
    print("FILE:", fp.name)
    print("=" * 72)
    out = parser.process_fasta(fp)
    results[fp.name] = out
    pprint_result(out)

results


FILE: uniprot_download.fasta
{
  "seqkit": {
    "ok": true,
    "molecule_type": "Protein",
    "stats": {
      "file": "uniprot_download.fasta",
      "format": "FASTA",
      "type": "Protein",
      "num_seqs": "7",
      "sum_len": "3861",
      "min_len": "180",
      "avg_len": "551.6",
      "max_len": "1382"
    }
  },
  "sequences": [
    {
      "description": "sp|Q9R1A7|NR1I2_RAT Nuclear receptor subfamily 1 group I member 2 OS=Rattus norvegicus OX=10116 GN=Nr1i2 PE=2 SV=1",
      "sequence": "MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLLPHLADVSTYMFKGVINFAKVISHFRELPIEDQISLLKGATFEMCILRFNTMFDTETGTWECGRLAYCFEDPNGGFQKLLLDPLMKFHCMLKKLQLREEEYVLMQAISLFSPDRPGVVQRSVVDQLQERFALTLKAYIECSRPYPAHRFLFLKIMAVLTELRSINAQQTQQLLRIQDTHPFATPLMQELFSSTDG",
      "db_name": "UniProt",
      "db_info": {
      

{'uniprot_download.fasta': {'seqkit': {'ok': True,
   'molecule_type': 'Protein',
   'stats': {'file': 'uniprot_download.fasta',
    'format': 'FASTA',
    'type': 'Protein',
    'num_seqs': '7',
    'sum_len': '3861',
    'min_len': '180',
    'avg_len': '551.6',
    'max_len': '1382'}},
  'sequences': [{'description': 'sp|Q9R1A7|NR1I2_RAT Nuclear receptor subfamily 1 group I member 2 OS=Rattus norvegicus OX=10116 GN=Nr1i2 PE=2 SV=1',
    'sequence': 'MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLLPHLADVSTYMFKGVINFAKVISHFRELPIEDQISLLKGATFEMCILRFNTMFDTETGTWECGRLAYCFEDPNGGFQKLLLDPLMKFHCMLKKLQLREEEYVLMQAISLFSPDRPGVVQRSVVDQLQERFALTLKAYIECSRPYPAHRFLFLKIMAVLTELRSINAQQTQQLLRIQDTHPFATPLMQELFSSTDG',
    'db_name': 'UniProt',
    'db_info': {'Q9R1A7': {'organism': 'Rattus norvegicus',
      'geneInfo': [{'ge